In [1]:
import pandas as pd
import os

folder = "data/raw"

for file in os.listdir(folder):

    if file.endswith(".csv"):

        print("\n" + "=" * 80)
        print(f"FILE: {file}")
        print("=" * 80)

        try:
            df = pd.read_csv(os.path.join(folder, file))

            # Shape
            print("\n[1] Shape")
            print(df.shape)

            # Data Types
            print("\n[2] Data Types")
            print(df.dtypes)

            # First 5 Rows
            print("\n[3] First 5 Rows")
            print(df.head())

            # Missing Values
            print("\n[4] Missing Values")
            print(df.isnull().sum())

            # Duplicate Rows
            print("\n[5] Duplicate Rows")
            print(df.duplicated().sum())

            # Column Names
            print("\n[6] Column Names")
            print(df.columns.tolist())

            # Summary Statistics
            print("\n[7] Summary Statistics")
            print(df.describe(include="all"))

            # Negative Values Check
            print("\n[8] Negative Value Check")

            numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

            for col in numeric_cols:
                negative_count = (df[col] < 0).sum()

                if negative_count > 0:
                    print(f"{col}: {negative_count} negative values found")

            # Date Validation
            print("\n[9] Date Column Validation")

            for col in df.columns:

                if "date" in col.lower():

                    converted = pd.to_datetime(
                        df[col],
                        errors="coerce"
                    )

                    invalid_dates = converted.isnull().sum()

                    print(
                        f"{col}: {invalid_dates} invalid dates"
                    )

            print("\n[10] Data Quality Notes")

            total_missing = df.isnull().sum().sum()
            total_duplicates = df.duplicated().sum()

            print(f"Total Missing Values : {total_missing}")
            print(f"Total Duplicate Rows : {total_duplicates}")

            if total_missing == 0:
                print("✓ No missing values")

            if total_duplicates == 0:
                print("✓ No duplicate rows")

        except Exception as e:
            print(f"Error reading {file}")
            print(e)

print("\n\nData Ingestion Completed Successfully")


FILE: 01_fund_master.csv

[1] Shape
(40, 15)

[2] Data Types
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object

[3] First 5 Rows
   amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fund - Regular

C:\Users\Dell\AppData\Local\Temp\ipykernel_23816\4026730963.py:63: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  converted = pd.to_datetime(


In [2]:
import requests
import pandas as pd
import os

# Create folder if it doesn't exist
os.makedirs("data/raw", exist_ok=True)

schemes = {
    "SBI_Bluechip": 119551,
    "ICICI_Bluechip": 120503,
    "Nippon_LargeCap": 118632,
    "Axis_Bluechip": 119092,
    "Kotak_Bluechip": 120841
}

for name, code in schemes.items():

    url = f"https://api.mfapi.in/mf/{code}"

    response = requests.get(url)

    if response.status_code == 200:

        data = response.json()

        nav_df = pd.DataFrame(data["data"])

        file_path = f"data/raw/{name}.csv"

        nav_df.to_csv(file_path, index=False)

        print(f"✓ {name} saved successfully")

    else:
        print(f"✗ Failed to fetch {name}")

✓ SBI_Bluechip saved successfully
✓ ICICI_Bluechip saved successfully
✓ Nippon_LargeCap saved successfully
✓ Axis_Bluechip saved successfully
✓ Kotak_Bluechip saved successfully
